<a href="https://colab.research.google.com/github/mazin04-a11y/Agentic-AI-Memory/blob/main/ThreatCrew_%E2%80%94_A_Multi_Agent_AI_Security_Investigation_System_By_Mazin_Abdel_Hameed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ThreatCrew — A Multi-Agent AI Security Investigation System

## Table of Contents

1. Project Overview  
2. System Architecture  
3. Agent Roles  
4. Investigation Workflow  
5. Security Safeguards  
6. Implementation Stages  
7. Code Execution Overview  

---

<div style="border:3px solid #999;padding:30px;border-radius:12px;background:#fafafa">

# Project Overview

This project implements a **hierarchical multi-agent security investigation system** using **CrewAI**, **SQLite**, and **live external threat intelligence**.

Instead of relying on a single language model response, the system simulates a **coordinated cybersecurity investigation team** where each AI agent performs a specialized role.

The system is capable of:

• retrieving verified security data from an internal database  
• interpreting historical threat intelligence patterns  
• checking live external IP reputation  
• producing a structured professional security report  
• validating the report through an automated quality-control agent  

---

> ### Key Idea
>
> A single AI answering a question can hallucinate.
>
> A **team of specialized AI agents with tools** can investigate and verify evidence.

---

# System Architecture

The system follows a **hierarchical agent structure**.

The Crew Manager coordinates specialist agents.

System Structure

<pre>
                     USER
                       │
                       ▼
                Crew Manager
                       │
        ┌──────────────┼──────────────┐
        │              │              │
        ▼              ▼              ▼
  Security Analyst  Incident Historian  Reputation Analyst
        (SQL)        (Threat File)        (AbuseIPDB)
        │              │              │
        └──────────────┼──────────────┘
                       ▼
              Draft Security Report
                       │
                       ▼
          Security Assessment Validator
                       │
              APPROVED or REJECTED
</pre>

---

> ### Why This Architecture Works
>
> Each agent has **limited responsibilities and tools**, forcing the system to rely on real evidence instead of guessing.

---

# Agent Roles

Each agent is engineered with **role, goal, and backstory design**.

This encourages the model to behave like a **specialist investigator**.

---

## Security Analyst

Responsible for retrieving **hard evidence from the SQL database**.

Example database query used during investigations:

SELECT COALESCE(SUM(attempt_count),0)  
FROM access_logs  
WHERE source_ip = 'IP_ADDRESS'  
AND UPPER(status) = 'FAILED'

---

> ### What This Means
>
> Numerical evidence used in investigations always comes directly from the **database query results**, ensuring grounded security analysis.

---

## Incident Historian

Reads the **threat intelligence vault** and analyzes historical attack patterns.

Example intelligence rule stored in the system:

More than **10 failed login attempts from a single IP**  
should be classified as a **Critical Alert**.

---

> ### Historical Interpretation
>
> The historian determines whether the observed behavior  
> matches previously documented attack patterns.

---

## IP Reputation Analyst

Retrieves **live external threat intelligence** using a reputation API such as AbuseIPDB.

Typical information retrieved may include:

• abuse confidence score  
• number of abuse reports  
• geographic origin of the IP  
• hosting provider or network type  

---

> ### Reputation Interpretation
>
> External reputation signals provide **additional context** that complements the internal security evidence.

---

## Security Assessment Validator

The validator agent performs **automated quality assurance**.

The validator checks that:

• SQL numbers match query results  
• threat intelligence context is represented correctly  
• external reputation data is interpreted carefully  
• conclusions are supported by evidence  

---

> ### Self-Correction Mechanism
>
> If the validator finds a problem:
>
> REJECTED → investigation reruns
>
> If everything is correct:
>
> APPROVED → final report delivered

---

# Investigation Workflow

The investigation follows a structured pipeline.

<pre>
User Investigation Query
        │
        ▼
Manager analyzes request
        │
        ▼
Manager delegates work
        │
        ▼
Specialist agents gather evidence
        │
        ▼
Manager synthesizes report
        │
        ▼
Validator reviews assessment
        │
   ┌────┴────┐
   ▼         ▼
APPROVED   REJECTED
   │         │
   ▼         ▼
Final     Revision
Report      Loop
</pre>

---

> ### Important Concept
>
> The system mirrors a **Security Operations Center investigation workflow**, where multiple analysts collaborate to reach a conclusion.

---

# Security Safeguards

The SQL tool enforces **read-only database access**.

Blocked commands include:

DROP  
DELETE  
UPDATE  
INSERT  
ALTER  
TRUNCATE  

Only **SELECT queries** are permitted.

---

> ### Why This Matters
>
> The investigation system **cannot modify evidence**.  
> It can only retrieve and analyze existing security logs.

---

# Implementation Stages

The system was developed in several structured phases.

---

### Stage 1 — Secure Environment Setup

API keys are loaded from secure storage and CrewAI dependencies are installed.

---

### Stage 2 — Data Sources

Two knowledge layers are created.

Internal Security Logs  
→ SQLite Database

Threat Intelligence Vault  
→ Local threat intelligence file

---

### Stage 3 — Tool Creation

Custom tools allow agents to safely access:

• the SQL database  
• threat intelligence files  
• live reputation APIs  

---

### Stage 4 — Agent Engineering

Agents are configured with:

• specialized roles  
• investigation rules  
• restricted tool access  

---

### Stage 5 — Hierarchical Crew

The Crew Manager dynamically coordinates specialist agents during investigations.

---

### Stage 6 — Validation Loop

A validator agent reviews the final assessment before approval.

---

# Code Execution Overview

When the program runs, the investigation process looks like this.

<pre>
User enters investigation query
        │
        ▼
Crew Manager analyzes request
        │
        ▼
Security Analyst retrieves SQL evidence
Incident Historian retrieves threat intelligence
Reputation Analyst retrieves external signals
        │
        ▼
Manager synthesizes draft report
        │
        ▼
Validator verifies evidence
        │
        ▼
Final approved security assessment
</pre>

---

> ### Final Outcome
>
> The system transforms a language model into a **collaborative AI cybersecurity investigation team** capable of retrieving evidence, reasoning about threats, and validating its own conclusions.

</div>

In [3]:
## ---- Stage 1: Secure Environment & Scaffold (Weeks 1–3) ----
## ----- Per the security protocols, we use google.colab.userdata to manage
## ----- credentials without hardcoding them.

# Installation of the core orchestration framework (Week 4)
!pip install crewai crewai_tools requests

import os
import sqlite3
import requests
from google.colab import userdata
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
from crewai_tools import FileReadTool

# Secure credential retrieval (Week 3)
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["SERPER_API_KEY"] = userdata.get("SERPER_API_KEY")

try:
    os.environ["ABUSEIPDB_API_KEY"] = userdata.get("ABUSEIPDB_API_KEY")
except Exception:
    print("ABUSEIPDB_API_KEY not found. Live IP reputation checks will be disabled.")


## ---- Stage 2: The Relational Foundation (Week 14) ----
# We initialize the SQLite database, which acts as the agent's persistent
# "hard drive" for structured security facts. We also create a contextual
# threat intelligence file to simulate the "semantic vault" for qualitative memory.

def setup_security_db():
    # Creating the local 'security_ops.db' file (The Security System's Hard Drive)
    conn = sqlite3.connect("security_ops.db")
    cursor = conn.cursor()

    # Initializing the structured access log table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS access_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            source_ip TEXT,
            attempt_count INTEGER,
            status TEXT
        )
    """)

    # Safety reset to avoid duplicate rows on notebook re-runs
    cursor.execute("DELETE FROM access_logs")

    access_log_data = [
        ("192.168.1.50", 12, "FAILED"),
        ("10.0.0.15", 1, "SUCCESS"),
        ("45.155.205.233", 15, "FAILED")
    ]

    cursor.executemany(
        "INSERT INTO access_logs (source_ip, attempt_count, status) VALUES (?, ?, ?)",
        access_log_data
    )

    conn.commit()
    conn.close()


def setup_threat_intel_file():
    # Creating a small contextual threat intelligence file (The Semantic Vault)
    intel_text = """
Known Threat Actors:
- The IP range 192.168.x.x has been associated with brute force testing in the past.
- Previous incident report (Feb 2026): A spike of more than 10 failed attempts from a single IP is classified as a Critical Alert.
- Repeated FAILED access attempts from the same source IP should be treated as suspicious until reviewed.
- Public scanner and abuse-host behavior may appear in live external reputation feeds even when internal context is limited.
"""

    with open("threat_intel.txt", "w", encoding="utf-8") as f:
        f.write(intel_text)


setup_security_db()
setup_threat_intel_file()


## ---- Stage 3: Safety-First Custom Tooling (Weeks 7 & 14) ----
# Following the Best Practice, we inherit from BaseTool to ensure strict
# validation. We implement a Safety Safeguard to prevent destructive SQL commands.
# We also add a live external reputation tool to extend the assistant beyond
# local simulation into real-time IP intelligence retrieval.

class SafeQueryTool(BaseTool):
    name: str = "Database Query Tool"
    description: str = (
        "Executes read-only SQL queries to fetch grounded facts from the "
        "access_logs table."
    )

    def _run(self, query: str) -> str:
        # Mandatory Safety Safeguard: Prohibiting destructive actions (Week 14)
        if any(cmd in query.upper() for cmd in ["DROP", "DELETE", "UPDATE", "ALTER", "TRUNCATE", "INSERT"]):
            raise ValueError("Action Prohibited: Read-Only Access Only")

        try:
            conn = sqlite3.connect("security_ops.db")
            cursor = conn.cursor()
            cursor.execute(query)
            results = cursor.fetchall()
            conn.close()
            return f"Retrieved data: {results}"
        except Exception as e:
            return f"Error executing query: {e}"


class AbuseIPDBTool(BaseTool):
    name: str = "IP Reputation Tool"
    description: str = (
        "Checks live external IP reputation data using AbuseIPDB. "
        "Use this tool when the mission requires current real-world IP threat intelligence."
    )

    def _run(self, ip_address: str) -> str:
        api_key = os.environ.get("ABUSEIPDB_API_KEY")

        if not api_key:
            return "Error: ABUSEIPDB_API_KEY is not set. Live reputation checks are unavailable."

        url = "https://api.abuseipdb.com/api/v2/check"
        headers = {
            "Key": api_key,
            "Accept": "application/json"
        }
        params = {
            "ipAddress": ip_address,
            "maxAgeInDays": 90
        }

        try:
            response = requests.get(url, headers=headers, params=params, timeout=20)
            response.raise_for_status()
            data = response.json().get("data", {})

            result = {
                "ipAddress": data.get("ipAddress"),
                "abuseConfidenceScore": data.get("abuseConfidenceScore"),
                "countryCode": data.get("countryCode"),
                "usageType": data.get("usageType"),
                "isp": data.get("isp"),
                "domain": data.get("domain"),
                "totalReports": data.get("totalReports"),
                "lastReportedAt": data.get("lastReportedAt")
            }

            return f"Live reputation data: {result}"
        except Exception as e:
            return f"Error checking live IP reputation: {e}"


sql_tool = SafeQueryTool()
read_tool = FileReadTool(file_path="threat_intel.txt")
ip_reputation_tool = AbuseIPDBTool()


## ---- Stage 4: Deep Role Engineering (Weeks 4, 11 & 14) ----
# We define specialized agents using the Role, Goal, and Backstory pattern.
# In the hierarchical model, the Crew Manager acts as the "Boss" that
# intelligently delegates the mission to the correct specialist.

# The Fact Path: Security Analyst Agent
analyst = Agent(
    role="Security Analyst",
    goal="Retrieve exact operational security facts from the SQL database without hallucination.",
    backstory=(
        "You are a precision-focused security analyst who only provides grounded facts "
        "from structured SQL data. You never invent incident counts or statuses.\n\n"
        "Database schema:\n"
        "- access_logs(id, source_ip, attempt_count, status)\n\n"
        "Valid status values:\n"
        "- FAILED\n"
        "- SUCCESS\n\n"
        "Rules:\n"
        "- Use ONLY this table and these column names.\n"
        "- Always write valid SELECT queries.\n"
        "- Use the exact status values as stored in the database.\n"
        "- When filtering status values, prefer robust queries such as UPPER(status) = 'FAILED'.\n"
        "- When using SUM(attempt_count), use COALESCE(SUM(attempt_count), 0) so missing rows return 0 instead of NULL.\n"
        "- attempt_count stores the number of failed or successful attempts in a row.\n"
        "- If the user asks 'how many attempts', use SUM(attempt_count), not COUNT(*), unless they are explicitly asking for the number of rows.\n"
        "- Never guess column names.\n"
        "- If unsure about structure, inspect the schema before querying."
    ),
    tools=[sql_tool],
    verbose=True,
    allow_delegation=False
)

# The Context Path: Incident Historian Agent
researcher = Agent(
    role="Incident Historian",
    goal="Retrieve qualitative security context and historical threat intelligence from the semantic vault.",
    backstory=(
        "You specialize in reading contextual security memory and identifying useful "
        "background information from prior threat notes and internal intelligence files.\n\n"
        "Rules:\n"
        "- Only report threat patterns explicitly written in the threat_intel file.\n"
        "- Do not infer or invent historical links that are not directly supported by the file.\n"
        "- If the file does not directly link a specific IP to a known threat, say so clearly.\n"
        "- You may explain that internal failed-attempt behavior aligns with a threshold described in the file, but do not claim a direct historical record unless the file explicitly states it."
    ),
    tools=[read_tool],
    verbose=True,
    allow_delegation=False
)

# The Live Threat Path: IP Reputation Analyst Agent
reputation_analyst = Agent(
    role="IP Reputation Analyst",
    goal="Retrieve live external IP reputation data without hallucination.",
    backstory=(
        "You are a live threat intelligence specialist. You use an external IP reputation service "
        "to retrieve current abuse signals, report counts, and reputation indicators.\n\n"
        "Rules:\n"
        "- Only report values returned by the live reputation tool.\n"
        "- Do not invent scores, reports, or threat labels.\n"
        "- If the tool fails, report the failure clearly.\n"
        "- If the IP address is private or reserved, explain that public threat reputation may be limited or not fully meaningful.\n"
        "- Use this path only for live external threat intelligence."
    ),
    tools=[ip_reputation_tool],
    verbose=True,
    allow_delegation=False
)

# The Quality Control Path: Security Assessment Validator
validator = Agent(
    role="Security Assessment Validator",
    goal="Validate that the final security assessment is grounded, complete, and consistent.",
    backstory=(
        "You are a strict quality control validator. You review the draft security assessment "
        "and check whether every claim is supported by grounded specialist outputs.\n\n"
        "Validation Rules:\n"
        "- Approve only if numerical values match the SQL results.\n"
        "- Approve only if historical threat context is represented accurately.\n"
        "- Approve only if live external reputation is interpreted carefully.\n"
        "- Do not reject a date merely because it looks recent; only reject it if it is explicitly impossible or unsupported.\n"
        "- If a live reputation tool returns a recent report date, treat it as valid tool output unless it clearly conflicts with the current date.\n"
        "- Do not require the draft to quote raw SQL tuples verbatim; it is enough that the numerical claim matches the SQL result.\n"
        "- If the IP is private or reserved, ensure the draft explains that external reputation may be limited.\n"
        "- Reject if the draft overstates or invents facts.\n"
        "- Reject if the final recommendation is vague, inconsistent, or unsupported.\n"
        "- Return exactly one of these formats:\n"
        "  APPROVED: <short reason>\n"
        "  REJECTED: <short reason and what must be fixed>"
    ),
    verbose=True,
    allow_delegation=False
)


## ---- Stage 5: Task Orchestration, Validation & Memory Activation (Weeks 5, 9 & 14) ----
# In Week 14, the hierarchical process is the correct architecture for
# true routing and selection. The manager LLM decides which specialist
# should act, and can combine all specialists when a mission needs
# grounded security facts, historical threat context, and live external reputation.
# We now add the course-aligned self-correcting validation loop to ensure
# the final answer is approved before the mission is complete.

# The Main Mission: Dynamic Routing + Draft Security Assessment
assistant_mission = Task(
    description="""
Answer the user query: '{user_input}'.

Revision Notes:
{revision_notes}

Mission Rules:
1. Determine whether the question requires:
   - hard facts from the SQL database,
   - contextual memory from the threat intelligence vault,
   - live external IP reputation data,
   - or a combination of all three.
2. Delegate factual retrieval to the Security Analyst.
3. Delegate qualitative or historical threat context retrieval to the Incident Historian.
4. Delegate live external threat reputation retrieval to the IP Reputation Analyst when the mission asks for current real-world IP risk.
5. Synthesize the specialist outputs into one professional draft security assessment.
6. If a risk judgment is requested, explicitly explain whether the facts match known threat patterns.
7. Do not hallucinate values that are not returned by a specialist tool.
8. If revision notes are provided, correct the draft accordingly before finalizing it.
9. When using live reputation data, treat tool-returned fields as the source of truth.
10. When citing internal failed attempts, ensure the numerical claim matches the SQL result.

Helpful grounded SQL examples:
- SELECT attempt_count FROM access_logs WHERE source_ip = '192.168.1.50';
- SELECT status FROM access_logs WHERE source_ip = '192.168.1.50';
- SELECT COALESCE(SUM(attempt_count), 0) FROM access_logs WHERE source_ip = '192.168.1.50' AND UPPER(status) = 'FAILED';

Known database values:
- status values are stored as uppercase strings: FAILED, SUCCESS
- attempt_count stores the number of attempts in a row, so totals should use SUM(attempt_count)

Live reputation guidance:
- Use the IP Reputation Analyst when the mission asks whether the IP is malicious, abusive, suspicious, or externally reported.
- Treat the live reputation tool as external real-world threat intelligence, separate from internal logs.
- Do not overstate external maliciousness when abuse scores are low or modest.
""",
    expected_output="A professional draft security recommendation derived from structured SQL facts, threat intelligence context, and live external reputation data when needed."
)

# The Validation Mission: Quality Control Review
validation_task = Task(
    description="""
Review the following draft security assessment for the user query.

User Query:
{user_input}

Draft Security Assessment:
{draft_answer}

Validation Checklist:
1. Are all numerical values grounded in SQL results?
2. Is the historical threat context represented accurately?
3. Is the live external reputation interpreted correctly?
4. If the IP is private or reserved, does the draft explain that external reputation may be limited?
5. Is the final recommendation supported by the evidence?
6. Does the draft avoid hallucinated or unsupported claims?

Return exactly one of these formats:
APPROVED: <short reason>
REJECTED: <short reason and what must be fixed>
""",
    expected_output="A validation result beginning with either APPROVED or REJECTED.",
    agent=validator
)

# The Validation Crew: Editor-style approval check
validation_crew = Crew(
    agents=[validator],
    tasks=[validation_task],
    process=Process.sequential,
    verbose=True
)

# The Final Result: A Stateful Security Intelligence with Self-Correction
user_query = str(input("Enter a security investigation query: "))

max_retries = 3
attempt = 0
approved = False
draft_answer = ""
revision_notes = "No previous revision notes."

while attempt < max_retries and not approved:
    print(f"\n--- Draft Attempt {attempt + 1} ---\n")

    # Recreate the hierarchical crew on every retry to avoid
    # the CrewAI manager-tools persistence bug on repeated hierarchical runs.
    drafting_crew = Crew(
        agents=[analyst, researcher, reputation_analyst],
        tasks=[assistant_mission],
        process=Process.hierarchical,   # <-- Enables true branching/selection
        manager_llm="gpt-4o",           # <-- The implicit Manager acts as the "Boss"
        manager_tools=[],               # <-- Prevents manager tool inheritance on retries
        memory=True,                    # <-- Activating the Semantic Memory Layer
        embedder={
            "provider": "openai",
            "config": {"model": "text-embedding-3-small"}
        },
        verbose=True
    )

    draft_answer = drafting_crew.kickoff(
        inputs={
            "user_input": user_query,
            "revision_notes": revision_notes
        }
    )

    print("\nDraft Security Assessment:\n")
    print(draft_answer)

    validation_result = validation_crew.kickoff(
        inputs={
            "user_input": user_query,
            "draft_answer": str(draft_answer)
        }
    )

    print("\nValidation Result:\n")
    print(validation_result)

    if str(validation_result).strip().startswith("APPROVED"):
        approved = True
    else:
        revision_notes = str(validation_result)
        attempt += 1

if approved:
    print("\nFinal Approved Security Assessment:\n")
    print(draft_answer)
else:
    print("\nValidation failed after maximum retries.")


Enter a security investigation query: Perform a full security investigation for IP address 45.155.205.233.  Use the internal security database to determine how many failed login attempts have occurred.  Then analyze the threat intelligence file to determine whether this activity matches known attack patterns.  Next check the live external reputation of the IP address.  Compare internal evidence with external threat intelligence and explain whether the behavior is consistent with a brute-force attack.  Finally classify the overall security risk as LOW, MEDIUM, or HIGH and justify the conclusion using only verified evidence.

--- Draft Attempt 1 ---



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 160cbaf1-0901-42c9-8cef-012c3a474205                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Answer the user query: 'Perform a full security investigation for IP address 45.155.205.233.  Use the          │
│  internal security database to determine how many failed login attempts have occurred.  Then analyze the        │
│  threat intelligence file to determine whether this activity matches known attack patterns.  Next check the     │
│  live external reputation of the IP address.  Compare internal evidence with external threat intelligence and   │
│  explain whether the behavior is consistent with a brute-force attack.  Finally classify the overall security   │
│  risk as LOW, MEDIUM, or HIGH and justify the conclusion using only verified evidence.'.                        │
│                                                                                                                 │
│  Revision Notes:                                                                                                │
│  No previous revision notes.                                                                                    │
│                                                                                                                 │
│  Mission Rules:                                                                                                 │
│  1. Determine whether the question requires:                                                                    │
│     - hard facts from the SQL database,                                                                         │
│     - contextual memory from the threat intelligence vault,                                                     │
│     - live external IP reputation data,                                                                         │
│     - or a combination of all three.                                                                            │
│  2. Delegate factual retrieval to the Security Analyst.                                                         │
│  3. Delegate qualitative or historical threat context retrieval to the Incident Historian.                      │
│  4. Delegate live external threat reputation retrieval to the IP Reputation Analyst when the mission asks for   │
│  current real-world IP risk.                                                                                    │
│  5. Synthesize the specialist outputs into one professional draft security assessment.                          │
│  6. If a risk judgment is requested, explicitly explain whether the facts match known threat patterns.          │
│  7. Do not hallucinate values that are not returned by a specialist tool.                                       │
│  8. If revision notes are provided, correct the draft accordingly before finalizing it.                         │
│  9. When using live reputation data, treat tool-returned fields as the source of truth.                         │
│  10. When citing internal failed attempts, ensure the numerical claim matches the SQL result.                   │
│                                                                                                                 │
│  Helpful grounded SQL examples:                                                                                 │
│  - SELECT attempt_count FROM access_logs WHERE source_ip = '192.168.1.50';                                      │
│  - SELECT status FROM access_logs WHERE source_ip = '192.168.1.50';                                             │
│  - SELECT COALESCE(SUM(attempt_count), 0) FROM access_l

╭────────────────────────────────────────────── 🧠 Memory Retrieval ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Started                                                                                       │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieved ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Completed                                                                                     │
│  Time: 2420.53ms                                                                                                │
│  Content:                                                                                                       │
│  Relevant memories:                                                                                             │
│  - (score=0.86) There have been a total of 3 abuse reports against the IP 45.155.205.233.                       │
│    categories: security, networking, incident report, network monitoring, security alert                        │
│    entities: ['45.155.205.233']                                                                                 │
│    dates: []                                                                                                    │
│    topics: ['abuse report', 'network security']                                                                 │
│  - (score=0.86) Ongoing monitoring is recommended for IP 45.155.205.233 to mitigate potential risks.            │
│    categories: security, networking, network monitoring, security alert                                         │
│    entities: ['IP 45.155.205.233']                                                                              │
│    dates: []                                                                                                    │
│    topics: ['risk mitigation', 'network monitoring']                                                            │
│  - (score=0.85) The activity of repeated failed login attempts for IP 45.155.205.233 aligns with general        │
│  suspicious patterns noted in security alerts.                                                                  │
│    categories: security, networking, incident report, network monitoring, security alert                        │
│    entities: ['IP 45.155.205.233']                                                                              │
│    dates: []                                                                                                    │
│    topics: ['failed login attempts', 'suspicious patterns', 'security alerts']                                  │
│  - (score=0.85) The source IP 45.155.205.233 has an abuse confidence score of 7, is classified as a Medium      │
│  Security Risk based on internal activity and external abuse reports, but does not show a high level of         │
│  malicious activity at this time.                                                                               │
│    categories: security, networking, incident report, network monitoring, security alert                        │
│    entities: ['45.155.205.233']                                                                                 │
│    dates: []                                                                                                    │
│    topics: ['abuse confidence score', 'IP address monitoring']                                                  │
│  - (score=0.84) The source IP 45.155.205.233 has recorded a total of 15 failed login attempts in the internal   │
│  security logs.                                                                                                 │
│    categories: security, incident report, security alert, network monitoring                                    │
│    entities: ['45.155.205.233']                                                                                 │
│    dates: []                                                                                                    │
│    topics: ['failed security attempts', 'internal secur

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Security Assessment Validator                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  APPROVED: Numerical values for failed attempts and abuse reports align with query results; historical context  │
│  about brute-force patterns and thresholds is accurate; live reputation interpretation is careful and           │
│  consistent; IP is public with correct external reputation references; final medium risk classification is      │
│  well supported by combined evidence without overstating facts.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Review the following draft security assessment for the user query.                                             │
│                                                                                                                 │
│  User Query:                                                                                                    │
│  Perform a full security investigation for IP address 45.155.205.233.  Use the internal security database to    │
│  determine how many failed login attempts have occurred.  Then analyze the threat intelligence file to          │
│  determine whether this activity matches known attack patterns.  Next check the live external reputation of     │
│  the IP address.  Compare internal evidence with external threat intelligence and explain whether the behavior  │
│  is consistent with a brute-force attack.  Finally classify the overall security risk as LOW, MEDIUM, or HIGH   │
│  and justify the conclusion using only verified evidence.                                                       │
│                                                                                                                 │
│  Draft Security Assessment:                                                                                     │
│  Based on the completed security investigation for IP address 45.155.205.233, here is the professional draft    │
│  security assessment:                                                                                           │
│                                                                                                                 │
│  **Security Investigation Summary:**                                                                            │
│                                                                                                                 │
│  1. **Internal Database Findings:**                                                                             │
│     - The IP address 45.155.205.233 has recorded a total of 15 failed login attempts in the internal security   │
│  logs. This exceeds the threshold of 10 failed attempts, which classifies it as a Critical Alert according to   │
│  past incident reports.                                                                                         │
│                                                                                                                 │
│  2. **Threat Intelligence Analysis:**                                                                           │
│     - The pattern of repeated failed login attempts aligns with suspicious behavior consistent with potential   │
│  brute-force attack patterns. While the specific IP is not explicitly linked to known brute-force attack        │
│  ranges, the volume and nature of the attempts suggest brute-force-like activity.                               │
│                                                                                                                 │
│  3. **Live External Reputation:**                                                                               │
│     - The live external reputation rates the IP with an abuse confidence score of 7, indicating a moderate      │
│  level of confidence in the abuse. It has been reported 3 times recently for abuse, and it is associated with   │
│  usage by a data center/web hosting service in Russia.                                                          │
│                                                        


Validation Result:

APPROVED: Numerical values for failed attempts and abuse reports align with query results; historical context about brute-force patterns and thresholds is accurate; live reputation interpretation is careful and consistent; IP is public with correct external reputation references; final medium risk classification is well supported by combined evidence without overstating facts.

Final Approved Security Assessment:

Based on the completed security investigation for IP address 45.155.205.233, here is the professional draft security assessment:

**Security Investigation Summary:**

1. **Internal Database Findings:**
   - The IP address 45.155.205.233 has recorded a total of 15 failed login attempts in the internal security logs. This exceeds the threshold of 10 failed attempts, which classifies it as a Critical Alert according to past incident reports.

2. **Threat Intelligence Analysis:**
   - The pattern of repeated failed login attempts aligns with suspicious behavi

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ecac34fc-36e2-407a-850b-5fdf3fd00c64                                                                       │
│  Final Output: APPROVED: Numerical values for failed attempts and abuse reports align with query results;       │
│  historical context about brute-force patterns and thresholds is accurate; live reputation interpretation is    │
│  careful and consistent; IP is public with correct external reputation references; final medium risk            │
│  classification is well supported by combined evidence without overstating facts.                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# Output Explanation — Multi-Agent Investigation Execution

## Table of Contents

1. Investigation Trigger  
2. Manager Agent Analysis  
3. Specialist Agent Delegation  
4. Evidence Retrieval  
5. Draft Security Assessment  
6. Validator Review Process  
7. Final Approved Investigation Report  

---

<div style="border:3px solid #999;padding:30px;border-radius:12px;background:#fafafa">

# Investigation Trigger

The execution begins when the user enters a **security investigation query**.

Example query used in this run:

Investigate IP address **45.155.205.233** and determine:

• how many failed login attempts occurred  
• whether the activity matches known threat patterns  
• the live external reputation of the IP  
• the overall security risk classification  

---

> ### What Happens First
>
> The system does **not immediately answer the question**.
>
> Instead, the **Crew Manager agent analyzes the request** and determines what types of evidence must be collected.

---

# Manager Agent Analysis

The **Crew Manager** reads the investigation request and determines which specialist agents must participate.

The manager identifies three required evidence sources:

1. Internal security logs  
2. Historical threat intelligence  
3. Live external reputation data  

Decision process

<pre>
User Investigation Query
            │
            ▼
       Crew Manager
            │
   ┌────────┼─────────┐
   │        │         │
   ▼        ▼         ▼
SQL Data  Threat    External
Needed?   Context?  Reputation?
   │        │         │
  YES      YES       YES
   │        │         │
   ▼        ▼         ▼
Security  Incident  Reputation
Analyst   Historian   Analyst
</pre>

---

> ### Key Learning Point
>
> The manager **does not gather evidence itself**.
>
> It acts like a **team leader**, assigning tasks to specialist agents.

---

# Specialist Agent Delegation

The manager now delegates tasks to the investigation team.

Security Analyst  
Retrieve failed login attempts from the SQL database.

Incident Historian  
Determine whether the activity matches known attack patterns in the threat intelligence file.

IP Reputation Analyst  
Retrieve live external reputation signals for the IP address.

---

> ### Why Delegation Matters
>
> Each agent has **limited tools and responsibilities**, which prevents the system from hallucinating information.

---

# Evidence Retrieval

Each specialist agent now gathers evidence using its tools.

---

## Security Analyst — SQL Evidence

The Security Analyst queries the internal security database.

SQL query executed during the investigation:

SELECT COALESCE(SUM(attempt_count),0)  
FROM access_logs  
WHERE source_ip = '45.155.205.233'  
AND UPPER(status) = 'FAILED'

Database result returned:

[(15,)]

Interpretation:

15 failed login attempts were recorded for this IP.

---

> ### Evidence Source
>
> This number comes directly from the **security logs**, ensuring that the analysis is grounded in real data.

---

## Incident Historian — Threat Intelligence Context

The historian reads the **threat intelligence file**.

Relevant rule discovered:

More than **10 failed login attempts from a single IP**  
should be classified as a **Critical Alert**.

Observed evidence:

15 failed attempts

Comparison:

15 > 10

Interpretation:

The activity matches **suspicious authentication behavior consistent with brute-force login attempts**.

---

## IP Reputation Analyst — External Intelligence

The reputation analyst retrieves data from the AbuseIPDB API.

Signals returned during the investigation:

Abuse Confidence Score: 7  
Total Reports: 3  
Country: Russia (RU)  
Usage Type: Data Center / Hosting  

Interpretation:

The IP address has **moderate abuse reports**, but it is not strongly malicious.

---

> ### Evidence Summary

<pre>
Internal Evidence
15 failed login attempts

Threat Intelligence
Activity exceeds alert threshold

External Reputation
Moderate abuse reports detected
</pre>

---

# Draft Security Assessment

After collecting all evidence, the **Crew Manager synthesizes the findings**.

Reasoning process

<pre>
Internal Evidence
        +
Threat Intelligence
        +
External Reputation
        │
        ▼
   Risk Evaluation
</pre>

Manager conclusion:

The IP address demonstrates **suspicious login behavior**, but the external reputation does not indicate severe malicious activity.

Initial risk classification:

MEDIUM

---

> ### Important Concept
>
> The manager does not invent conclusions.
>
> It **combines the verified evidence retrieved by specialist agents**.

---

# Validator Review Process

Before the report is delivered, the system performs **automated quality assurance** using a Validator Agent.

Validator checks include:

• Are SQL values grounded in database results?  
• Is the threat intelligence interpretation accurate?  
• Is the external reputation data represented correctly?  
• Is the final risk classification supported by the evidence?  

Validation workflow

<pre>
Draft Security Assessment
           │
           ▼
      Validator Agent
           │
     ┌─────┴─────┐
     ▼           ▼
  APPROVED     REJECTED
     │           │
     ▼           ▼
Final Report  Revision Loop
</pre>

Validator decision in this execution:

APPROVED

Reason:

All numerical evidence matched the SQL results and the reasoning was consistent with the collected data.

---

# Final Approved Investigation Report

Security Investigation Summary

Internal Evidence  
The IP address **45.155.205.233** generated **15 failed login attempts** in the internal security logs.

Threat Intelligence  
This exceeds the **10-attempt alert threshold** defined in historical incident reports.

External Reputation  
The IP has:

Abuse Confidence Score: 7  
Total Abuse Reports: 3  
Country: Russia  
Hosting Type: Data Center  

These signals indicate **moderate abuse activity**.

---

# Risk Classification

Overall Risk Level → **MEDIUM**

---

# Justification

The MEDIUM classification is supported by three independent evidence sources.

<pre>
Internal Evidence
High number of failed login attempts

Threat Intelligence
Pattern matches suspicious login activity

External Reputation
Moderate abuse signals
</pre>

---

> ### Final Conclusion
>
> The IP address demonstrates **suspicious login activity requiring monitoring**,  
> but the available evidence does not confirm severe malicious behavior.

---

# What Students Should Learn From This Output

This execution demonstrates how a **multi-agent AI investigation system operates**.

Investigation pipeline

<pre>
User Query
     │
     ▼
Manager Analysis
     │
     ▼
Specialist Evidence Collection
     │
     ▼
Evidence Synthesis
     │
     ▼
Validator Quality Control
     │
     ▼
Final Security Assessment
</pre>

---

> ### Key Takeaway
>
> The system does not rely on a single AI response.
>
> Instead, it uses **collaborating agents, real tools, and validation** to produce reliable security assessments.

</div>